# 数据

In [1]:
import torch  # 导入PyTorch库，用于深度学习任务
from PIL import Image  # 导入PIL库中的Image模块，用于图像处理
import os  # 导入os库，用于处理文件和目录路径
from torch.utils.data import Dataset,DataLoader  # 从PyTorch的工具库中导入Dataset类，用于自定义数据集
import numpy as np

## 自定义数据集类

In [2]:
class ImageDataset(Dataset):
    #初始化
    def __init__(self,image_dir , transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.all_imgs = os.listdir(image_dir)

    def __len__(self):
        return len(self.all_imgs)
    
    def __getitem__(self, idx):  #根据索引获取图像
        img_loc = os.path.join(self.main_dir , self.all_imgs[idx])
        image = Image.open(img_loc).convert("RGB") #用PIL库打开图像并转换为RGB格式

        if self.transform is not None: #若定义了图像预处理
            tensor_image = self.transform(image) #对图像预处理，并转换为张量
        else:
            #若无变换，抛出异常提示必须提供预处理
            raise ValueError("transform 参数不能为None，需指定预处理方法")
        
        noise_factor  = 0.5

        noisy_imgs = tensor_image + noise_factor * torch.randn(*tensor_image.shape)
        noisy_imgs = np.clip(noisy_imgs, 0., 1.,)


In [ ]:
dataset = ImageDataset("../common/dataset/")
print(dataset.all_imgs)

## 定义模型

In [3]:
import torch.nn as nn
import torch.nn.functional as F 

In [7]:

class ConvDenoiser(nn.Module):
    def __init__(self):
        super(ConvDenoiser,self).__init__()

        ##编码器层
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,16,3,padding=1)
        self.conv3 = nn.Conv2d(16,8,3,padding=1)

        self.pool = nn.MaxPool2d(2,2)

        ## 解码器层
        #转置卷积层，卷积核为2，步幅为2 ，将空间维度增加2倍
        self.t_conv1 = nn.ConvTranspose2d(8,8,2,stride=2)
        self.t_conv2 = nn.ConvTranspose2d(8,16,2,stride=2)
        self.t_conv3 = nn.ConvTranspose2d(16,32,2,stride=2)

        #最后一个普通卷积，用于减少通道数
        self.conv_out = nn.Conv2d(32,3,3,padding=1)

    #前向传播
    def forward(self,x):
        ## 编码
        x = F.relu(self.conv1(x))
        print("conv1 shape:",x.shape)
        x = self.pool(x)
        print("pool1 shape:",x.shape)
        x = F.relu(self.conv2(x))
        print("conv2 shape:",x.shape)
        x = self.pool(x)
        print("pool2 shape:",x.shape)

        x = F.relu(self.conv3(x))
        print("conv3 shape",x.shape)
        x = self.pool(x) 
        print("pool3 shape:",x.shape)

        ## 解码
        #添加转置卷积层，带有ReLU激活函数
        x = F.relu(self.t_conv1(x))
        print("t_conv1 shape:",x.shape)
        x = F.relu(self.t_conv2(x))
        print("t_conv2 shape",x.shape)
        x = F.relu(self.t_conv3(x))
        print("t_conv3 shape",x.shape)

        x = F.sigmoid(self.conv_out(x))
        print("conv_out shape",x.shape)
        
        return x
    


In [8]:
    #初始神经网络
    model = ConvDenoiser()
    print(model)

    x = model.forward(torch.rand(1,3,64,64))

ConvDenoiser(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (t_conv1): ConvTranspose2d(8, 8, kernel_size=(2, 2), stride=(2, 2))
  (t_conv2): ConvTranspose2d(8, 16, kernel_size=(2, 2), stride=(2, 2))
  (t_conv3): ConvTranspose2d(16, 32, kernel_size=(2, 2), stride=(2, 2))
  (conv_out): Conv2d(32, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)
conv1 shape: torch.Size([1, 32, 64, 64])
pool1 shape: torch.Size([1, 32, 32, 32])
conv2 shape: torch.Size([1, 16, 32, 32])
pool2 shape: torch.Size([1, 16, 16, 16])
conv3 shape torch.Size([1, 8, 16, 16])
pool3 shape: torch.Size([1, 8, 8, 8])
t_conv1 shape: torch.Size([1, 8, 16, 16])
t_conv2 shape torch.Size([1, 16, 32, 32])
t_conv3 shape torch.Size([1, 32, 64, 64])
conv_out sh